# Projeto 1

Modelo de entrega da questão 2.

Respeite as divisões em Markdown para facilitar a navegação no relatório.

## Questão 2: Resolvendo sistemas tridiagonais.

### Questão 2.1
Escreva, em Python ou Julia, uma função `solve_tridiag(A, b)` que recebe uma matriz quadrada $A$ e um vetor $b$ e retorna a solução do sistema $Ax = b$ (você pode supor que $A$ é inversível).

explicação de pq o algoritmo de thomas e como ele funciona

In [1]:
def solve_tridiag(A, b):
    """
    Resolve sistema tridiagonal Ax = b usando o método de Thomas.
    A: matriz tridiagonal nxn
    b: vetor do lado direito - tamanho n
    """
    n = len(b)
 
    # Extrai as três diagonais de A
    inf  = [A[i][i-1] for i in range(1, n)]   # sub-diagonal
    diag = [A[i][i]   for i in range(n)]       # diagonal principal
    sup  = [A[i][i+1] for i in range(n-1)]     # super-diagonal
 
    # Criar cópias para não modificar os vetores originais
    sup_prime = [0.0] * (n - 1)
    b_prime   = [0.0] * n
    x         = [0.0] * n
 
 
    # Forward Sweep (fatoração)
    sup_prime[0] = sup[0] / diag[0]
    b_prime[0]   = b[0]  / diag[0]
 
    for i in range(1, n - 1):
        denom        = diag[i] - inf[i-1] * sup_prime[i-1]
        sup_prime[i] = sup[i] / denom
        b_prime[i]   = (b[i] - inf[i-1] * b_prime[i-1]) / denom
 
    b_prime[n-1] = (b[n-1] - inf[n-2] * b_prime[n-2]) / (diag[n-1] - inf[n-2] * sup_prime[n-2])
 
    # Back Substitution (substituição reversa)
    x[n-1] = b_prime[n-1]
 
    for i in range(n - 2, -1, -1):
        x[i] = b_prime[i] - sup_prime[i] * x[i+1]
 
    return x
 


### Questão 2.2
Verifique que sua função está correta, fazendo alguns testes.

Como b é combinação linear das colunas de A, podemos fazer essa combinação e gerar b a partir de um x qualquer. 
Obter esse x é o propósito da função solve_tridiag!
Gerar A e b de forma aleatória não faz sentido no contexto desse problema,  por isso criei a função gerar_sistema_tridiag. 

A função gerar_sistema_tridiag é criada apenas para fazer testes! 

O x produzido por gerar_sistema_tridiag é a solução do sistema, mas esse x é usado apenas para comparar com o resultado de solve_tridiag (similar a um gabarito).

In [ ]:
import time
import random


#medição de tempo das matrizes geradas aleatoriamente usando o perf_counter
def medir_tridiag(func, A, b, repeticoes=10):
    tempos = []
    for _ in range(repeticoes):
        inicio = time.perf_counter() #mensurado com perf_counter
        func(A, b)
        fim = time.perf_counter()
        tempos.append(fim - inicio)
    return sum(tempos) / len(tempos)


def gerar_sistema_tridiag(n):
    # gera x
    x = [random.uniform(1, 5) for _ in range(n)]

    # monta matriz tridiagonal
    A = [[0]*n for _ in range(n)] #só zeros

    for i in range(n):
        A[i][i] = random.uniform(5, 10)  # diagonal dom

        if i > 0:
            A[i][i-1] = random.uniform(1, 3) #diagonal inf

        if i < n-1:
            A[i][i+1] = random.uniform(1, 3) #diagonal sup

    # calcula b = Ax
    b = [0]*n
    for i in range(n):
        for j in range(n):
            b[i] += A[i][j]*x[j]

    return A, b, x


def rodar_teste(tamanhos, repeticoes=10):

    for n in tamanhos:
        A, b, x_real = gerar_sistema_tridiag(n)

        m = len(A)
        n = len(A[0])
        p = len(b) #verificar em solve_tridiag se o tamanho bate

        # x calculado em solve_tridiag
        x_calc = solve_tridiag(A, b)

        #exibir no terminal
        print(f"\n=== A: {m}×{n}  |  b: {m}×{1} ===")
        print(f"{'perf_counter':<15} {'solve_tridiag':>20}")
        #print("Calculado:", x_calc)
        #print("Esperado :", x_real)
        print("-" * 57)

        #medição
        t_tridiag = medir_tridiag(solve_tridiag, A, b, repeticoes)
        print(f"{t_tridiag:>20.6f}s")

tamanhos = [2, 3, 5, 10, 30, 50, 100, 200, 500, 1000, 2000, 5000, 10000, 20000]

rodar_teste(tamanhos, repeticoes=10)


=== A: 2×2  |  b: 2×1 ===
perf_counter           solve_tridiag
---------------------------------------------------------
            0.000009s

=== A: 3×3  |  b: 3×1 ===
perf_counter           solve_tridiag
---------------------------------------------------------
            0.000011s

=== A: 5×5  |  b: 5×1 ===
perf_counter           solve_tridiag
---------------------------------------------------------
            0.000021s

=== A: 10×10  |  b: 10×1 ===
perf_counter           solve_tridiag
---------------------------------------------------------
            0.000026s

=== A: 30×30  |  b: 30×1 ===
perf_counter           solve_tridiag
---------------------------------------------------------
            0.000064s

=== A: 50×50  |  b: 50×1 ===
perf_counter           solve_tridiag
---------------------------------------------------------
            0.000102s

=== A: 100×100  |  b: 100×1 ===
perf_counter           solve_tridiag
---------------------------------------------------------

### Questão 2.3
Qual é a complexidade computacional da sua função?

explicar pq ele é O(n)

### Questão 2.4
A partir de que tamanho de matriz $A$ a função `solve_tridiag` se torna mais rápida do que a função `numpy.linalg.solve` (ou `\`, em Julia)?
Como você fez para chegar neste resultado?

In [ ]:
import numpy as np

def solve_tridiag_np(A,b):
    np.linalg.solve(A, b)

def rodar_testes_np(limite, repeticoes=10):
    """
    compara solve_tridiag_np e solve_tridiag.
    usa só o medidor perf_counter
    """
    A, b, x_real = gerar_sistema_tridiag(n) 

    for n in range(limite, 100): #100 em 100
        # mesma matriz para todos (mesmos dados numéricos)
        A, b, x_real = gerar_sistema_tridiag(i)  

        m = len(A)
        n = len(A[0])   

        # aquecimento
        solve_tridiag(A,b)
        solve_tridiag_np(A,b)

        # medição
        t_np = medir_tridiag(solve_tridiag_np, A, b, repeticoes)
        t_tridiag = medir_tridiag(solve_tridiag, A, b, repeticoes)
        razao = t_np/t_tridiag

        #exibição no terminal
        print(f"\n=== A: {m}×{n}  |  b: {m}×{1} ===")
        print(f"{'Função':<20} {'Tempo (s)':>12} {'Razão vs solve_tridiag':>16}")
        print("-" * 50)
        print(f"{'solve_tridiag (lista)':<20} {t_tridiag:>12.6f} {'1.00x':>16}")
        print(f"{'solve_tridiag_np':<20} {t_np:>12.6f} {razao:>15.2f}x")


#CORRIGIR O FATO DE ACHAR O TEMPO T QUE ELES ULTRAPASSAM

rodar_testes_np(10000, repeticoes=10)

fazer grafico e tentar colocar ele aqui (corrigir o codigo).